# Dynamic Inventory Hedger: quickstart

This notebook runs the hedger in **paper mode** on live data. It seeds a sample position in a live Polymarket *"Bitcoin above $K"* market, then shows how FollowSM toxicity levels change the size and style of the Binance perp hedge.

You only need a FollowSM API key. Any tier works here, because the notebook uses `SIGNAL_SOURCE=poll`. ENTERPRISE adds the real-time WebSocket stream. [Get a key](https://follow-sm.com/pricing).

In [ ]:
!git clone -q https://github.com/Follow-SM/dynamic-inventory-hedger.git
%cd dynamic-inventory-hedger
!pip install -q -e .

In [ ]:
import os
from getpass import getpass

os.environ["FOLLOWSM_API_KEY"] = getpass("FollowSM API key: ")  # kept in memory only
os.environ["SIGNAL_SOURCE"] = "poll"
os.environ["JOURNAL_PATH"] = "journal.jsonl"

Run for 3 minutes. The default bands only hedge on genuinely toxic flow (`vpin_percentile ≥ 0.85`). On a calm market you will mostly see the **inventory limit** at work: any delta above `MAX_INVENTORY_USD` is hedged whatever the toxicity. To watch the passive PRE-hedge path, lower the band, e.g. `os.environ["PRE_HEDGE_PERCENTILE"] = "0.2"`.

In [ ]:
!python examples/polymarket_binance_hedge.py --minutes 3

In [ ]:
import pandas as pd

journal = pd.read_json("journal.jsonl", lines=True)
decisions = journal[journal.event == "decision"]
decisions[["ts", "level", "vpin_percentile", "poly_delta_usd", "hedge_usd", "imbalance", "mid"]].tail(20)

## Next steps

- Set `POLYMARKET_WALLET` to hedge your real positions: `!python -m dynamic_inventory_hedger`.
- Move to Binance **demo** trading with `LIVE_TRADING=true` and demo API keys (see the README).
- Real-time signals with the ENTERPRISE WebSocket: **[follow-sm.com/pricing](https://follow-sm.com/pricing)**